# 🚗 Synthetic Traffic Collision Dataset: Starter Notebook

## Introduction

Welcome to the starter notebook for the Synthetic CCTV-style Traffic Collision Dataset.
We provide a CARLA-based synthetic dataset that mimics fixed-camera accident videos—so you can train, debug, and iterate your pipeline before running it on the real test clips.
In this notebook, we will explore the dataset structure, parse the complex JSON annotations, and visualize the ground truth data by overlaying bounding boxes, tracking lines, and collision data directly onto the video sequences.

### Dataset details
* The dataset contains **2,211 videos**, simulated over **6 different maps**, using **14 different scenarios** in total (scenario stands for unique map location + accident type).
* Each scenario is simulated with multiple camera positions (up to ~50) and **5 distinct weather settings** (```rain```, ```clear```, ```sunset```, ```night```, and ```wet```).
* The simulation is dynamically generated -- for instance, the actors (types of vehicles, etc.) and their spawning locations are randomized, meaning there are no two same videos.

* The dataset is imbalanced, so the number of videos per accident type differ.
* The videos were manually filtered to remove wrong outcomes and simulation failures. However, it is possible that some videos or annotations may contain issues.

## What you learn

1. **Load & Inspect**: Understand the file structure and load metadata from labels.csv.

2. **Parse Annotations**: Handle GZipped JSON files to extract object detections, collision events, and sensor data.

3. **Map Data**: Align annotation timestamps with video frames.

4. **Visualize**: Render a fully annotated video showing object tracking and collision events in real-time.

# Step 1: Environment Setup

In [ ]:
import os
import os.path as osp
import json
import gzip
import yaml
import random
import cv2
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict, deque
from typing import Any

import sys
sys.path.append("../")
# Setting the path to the dataset
SIM_DATASET_PATH = "../runs/sim_dataset"
is_kaggle = False


### Utility Functions

The annotations are stored in compressed JSON format and YAML configuration files.

In [ ]:
def load_json(path: str, use_gzip: bool = True) -> Any:
    """Loads JSON/GZipped JSON annotations."""
    if use_gzip:
        with gzip.open(path, 'rt', encoding="ascii") as zipfile:
            return json.load(zipfile)
    with open(path, "r") as fp:
        return json.load(fp)


def load_yaml(path: str) -> Any:
    """Loads project configuration metadata."""
    with open(path, "r") as fp:
        return yaml.safe_load(fp)


# Step 2: Data Inspection
The dataset metadata is stored in labels.csv. This file acts as the central index, mapping video files to their annotations and providing high-level summary statistics for each scenario.

### DataFrame Schema

The sim_df dataframe contains the following columns:

| Category        | Column                   | Description                                                                                                          |
|-----------------|--------------------------|----------------------------------------------------------------------------------------------------------------------|
| Files           | rgb_path                 | Relative path to the raw video file (.mp4).                                                                          |
|                 | annotations_path         | Relative path to the compressed annotation file (.json.gz).                                                          |
| Scenario        | type                     | The category of accident simulated (e.g., t-bone, sideswipe).                                                        |
|                 | map                      | The CARLA town/map ID where the simulation took place.                                                               |
|                 | weather                  | Environmental conditions: clear, rain, wet, sunset, night.                                                           |
|                 | camera_position          | Unique identifier for the camera placement (viewpoint).                                                              |
| Timing          | accident_frame           | The exact frame index where the collision impact occurs.                                                             |
|                 | accident_time            | The timestamp (in seconds) of the collision.                                                                         |
|                 | duration                 | Total duration of the video clip in seconds.                                                                         |
|                 | no_frames                | Total number of frames in the video.                                                                                 |
| Synchronization | annotations_start_offset | The integer difference between the simulation _tick_ and the video frame 0. Used to align JSON data with MP4 frames. |
| Spatial         | center_x, center_y       | Normilzed pixel coordinates (x, y) representing the center of the collision impact.                                  |
|                 | x1, y1, x2, y2           | Normalized 2D bounding box coordinates encompassing the collision event.                                             |
| Video           | height, width            | Resolution of the video file (pixel dimensions).                                                                     |


In [ ]:
sim_df = pd.read_csv(osp.join(SIM_DATASET_PATH, "labels.csv"))
print(f"Total Videos: {len(sim_df)}")
sim_df.head()

### Dataset statistics

In [ ]:
sns.set_style("whitegrid")
fig, axes = plt.subplots(1, 3, figsize=(20, 5))

sns.countplot(y="type", hue="type", data=sim_df, ax=axes[0], order=sim_df['type'].value_counts().index, palette="viridis", legend=False)
axes[0].set_title("Distribution of Accident Types")
axes[0].set_xlabel("Video Count")
axes[0].set_ylabel("Accident Type")


sns.countplot(x="weather", hue="weather", data=sim_df, ax=axes[1], palette="magma", legend=False)
axes[1].set_title("Distribution of Weather Conditions")
axes[1].set_xlabel("Weather Type")
axes[1].set_ylabel("Video Count")

sns.countplot(x="map", hue="map", data=sim_df, ax=axes[2], palette="cubehelix", legend=False)
axes[2].set_title("Distribution of Simulation Maps")
axes[2].set_xlabel("Map Name")
axes[2].set_ylabel("Video Count")

plt.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(20, 5))

sns.histplot(
    data=sim_df,
    x="accident_time",
    kde=True,
    color="teal",
    bins=50
)

ax.set_title("Distribution of Accident Timing (Seconds into Video)", fontsize=14, pad=15)
ax.set_xlabel("Time of Accident (Seconds)", fontsize=12)
ax.set_ylabel("Number of Videos", fontsize=12)

mean_time = sim_df["accident_time"].mean()
ax.axvline(mean_time, color='red', linestyle='--', label=f'Mean: {mean_time:.2f}s')

median_time = sim_df["accident_time"].median()
ax.axvline(median_time, color='orange', linestyle='--', label=f'Median: {median_time:.2f}s')
ax.legend()

plt.tight_layout()
plt.show()


# Step 3: Annotations Parsing

The dataset provides three distinct layers of data within each JSON:

1. **Base** (```base```): Frame-by-frame 2D/3D bounding boxes and vehicle IDs.

2. **Collision** (```collision```): Specific frames identifying the accident.

3. **Sensor** (```sensor```): Static camera extrinsic data (Rotation/Location).

In [ ]:
# Select a random sample for visualization
sample_index = random.randint(0, len(sim_df) - 1)
sample = sim_df.iloc[sample_index]

video_path = osp.join(SIM_DATASET_PATH, sample["rgb_path"])

if is_kaggle:  # Kaggle unzips the .gz files implicitly
    annotations_path = osp.join(SIM_DATASET_PATH, sample["annotations_path"])
    annotations_path = annotations_path.removesuffix(".gz")
    annotations_path = osp.join(annotations_path, osp.basename(annotations_path))
else:
    annotations_path = osp.join(SIM_DATASET_PATH, sample["annotations_path"])

print(f"Annotation file path: {annotations_path}")

# Parse the annotation components
annotations = load_json(annotations_path, use_gzip=not is_kaggle)
base = annotations["base"]           # Frame object detections
collision = annotations["collision"] # Collision event
sensor = annotations["sensor"]       # Camera sensor placement

annotations_offset = sample["annotations_start_offset"]


We must subtract the ```annotations_start_offset``` from the simulation ```iteration``` to align data with video frame ```0```.

In [ ]:
# Map object detections to specific frames
frame_detections = {
    frame_annotations["iteration"] - annotations_offset: frame_annotations["objects"]
    for frame_annotations in base
}

# Map collision event to specific frames
collision_data = {}
for frame_collisions in collision:
    frame_index = frame_collisions["iteration"] - annotations_offset
    collision_data[frame_index] = {
        "collision_bbox": frame_collisions["collision_bbox"],
        "ids": frame_collisions["ids"],
    }


## Detection data

Detection data for a single frame consists of a list of detected objects. Each object represents one actor in the simulation that is visible from the camera.

Each detection contains the following fields:
- **id**:  Unique identifier of the actor in the simulation environment.
- **tag**:  Numeric class label of the actor.
- **location**: 3D coordinates (x, y, and z) of the center of the actor in the simulation world coordinate system.
- **extent**:  3D vector (x, y, and z)defining half the size of the actor's bounding box, measured from the actor's center to one of its vertices.
- **rotation**:  Orientation (pith, yaw, and roll angles) of the actor's 3D bounding box.
- **2d_bbox**: Pixel-space bounding box [[x1, y1], [x2, y2]] obtained by projecting the 3D bounding box to the 2D camera view plane.

The location, extent, and rotation fully describe the actor's position the 3D simulation environment.


In [ ]:
frame_detections[0]

## Collision data

Collision data describes frames in which a collision occurs, along with subsequent frames capturing the aftermath of the incident.

Each collision frame contains the following fields:
- **collision_bbox**:  Pixel-space circumferential bounding box [[x1, y1], [x2, y2]] with the actors involved in the incident.
- **ids**: List of unique identifiers of actors involved in the accident.

The collision_bbox may change over time as the involved actors move or separate following the impact.

In [ ]:
for accident_frame, accident_data in collision_data.items():
    print(f"Frame index", accident_frame)
    print(accident_data)
    break

## Sensor data

Sensor data provide information about the camera sensor position in the simulation world coordinate system.
The sensor object contains the following fields:
- **location**: 3D coordinates (x, y, and z) of the camera.
- **rotation**:  Orientation (pith, yaw, and roll angles) of the camera.


In [ ]:
sensor

## Tag-Class mapping

Load the file with tag to class mapping.

In [ ]:
cls_mapping = load_yaml(path=osp.join(SIM_DATASET_PATH, "annotation_classes.yaml"))
tag_to_cls_mapping = cls_mapping["names"]

# Step 4: Visualization: Rendering Annotations

The following function overlays three layers of information onto the video:

1. **Blue Boxes**: Standard object detections with class labels and IDs.
2. **RED Lines**: Unique object tracklets.
3. **Magenta Boxes**: Detected collision.

In [ ]:
def draw_tracks_on_video(
    video_path: str,
    frame_detections: list[dict[int, dict]],
    collision_data: list[dict[int, dict]],
    output_path: str,
    tag_to_cls_mapping: dict[int, str],
    thickness: int = 2,
    tracklet_length: int = 10000
) -> None:
    """
        Renders object detection bounding boxes, class labels, motion trajectories,
        and collision events onto a video file.

        Args:
            video_path: Path to the source MP4/RGB video file.
            frame_detections: Dictionary mapping frame indices to a list of object
                detections, where each detection contains '2d_bbox', 'id', and 'tag'.
            collision_data: Dictionary mapping frame indices to collision metadata,
                specifically 'collision_bbox' and involved actor 'ids'.
            output_path: Destination path for the annotated video file.
            tag_to_cls_mapping: Dictionary mapping integer class tags to
                human-readable string labels (e.g., {0: "Car", 1: "Pedestrian"}).
            thickness: Line thickness for bounding boxes and trajectory lines.
            tracklet_length: Maximum number of previous center-points to keep in
                memory for drawing motion tracklets.

        Returns:
            None. The result is written directly to the file specified in output_path.
        """
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise RuntimeError(f"Failed to open video: {video_path}")

    fps = cap.get(cv2.CAP_PROP_FPS)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    fourcc = cv2.VideoWriter_fourcc(*"avc1")
    writer = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

    track_history = defaultdict(lambda: deque(maxlen=tracklet_length))

    frame_idx = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break


        if 0 <= frame_idx < len(frame_detections):
            detections = frame_detections[frame_idx]

            for detection in detections:
                bbox = detection["2d_bbox"]
                track_id = detection["id"]
                tag = detection["tag"]

                x1, y1, x2, y2 = bbox[0][0], bbox[0][1], bbox[1][0], bbox[1][1]
                cx = int((x1 + x2) / 2)
                cy = int((y1 + (y2 - y1) * 0.8))
                track_history[track_id].append((cx, cy))

                # --- draw detection bbox (blue) ---
                cv2.rectangle(
                    frame,
                    (x1, y1),
                    (x2, y2),
                    color=(255, 0, 0), # BGR
                    thickness=thickness,
                )
                # --- draw tag label ---
                label = f"{tag_to_cls_mapping[tag]}: {track_id}"
                font = cv2.FONT_HERSHEY_SIMPLEX
                font_scale = 0.5
                font_thickness = 1


                (tw, th), baseline = cv2.getTextSize(
                    label, font, font_scale, font_thickness
                )
                # top-left of label background
                lx1 = x1
                ly1 = max(y1 - th - baseline - 4, 0)
                lx2 = x1 + tw + 4
                ly2 = ly1 + th + baseline + 4

                # label background
                cv2.rectangle(
                    frame,
                    (lx1, ly1),
                    (lx2, ly2),
                    color=(255, 0, 0),
                    thickness=-1,
                )

                # label text
                cv2.putText(
                    frame,
                    label,
                    (lx1 + 2, ly2 - baseline - 2),
                    font,
                    font_scale,
                    color=(255, 255, 255),
                    thickness=font_thickness,
                    lineType=cv2.LINE_AA,
                )

            if frame_idx in collision_data.keys():
                bbox = collision_data[frame_idx]["collision_bbox"]
                x1, y1, x2, y2 = bbox[0][0], bbox[0][1], bbox[1][0], bbox[1][1]

                cv2.rectangle(
                    frame,
                    (x1, y1),
                    (x2, y2),
                    color=(255, 0, 255),
                    thickness=thickness,
                )

        for track_id, points in track_history.items():
            if len(points) < 2:
                continue

            for i in range(1, len(points)):
                cv2.line(
                    frame,
                    points[i - 1],
                    points[i],
                    color=(0, 0, 255),
                    thickness=thickness,
                )

        writer.write(frame)
        frame_idx += 1

    cap.release()
    writer.release()

In [ ]:
annotated_video_output_path = osp.join(SIM_DATASET_PATH, "annotated.mp4")
draw_tracks_on_video(
    video_path=video_path,
    frame_detections=frame_detections,
    collision_data=collision_data,
    output_path=annotated_video_output_path,
    tag_to_cls_mapping=tag_to_cls_mapping,
)


In [ ]:
from IPython.display import Video, display


if osp.exists(annotated_video_output_path):
    display(Video(
        annotated_video_output_path,
        width=800,
        embed=True,
    ))
else:
    print("Video file not found. Check the output path.")